<a href="https://colab.research.google.com/github/maryamsohail32/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maryamsohail32/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
import os, sys, subprocess

REPO_URL = "https://github.com/maryamsohail32/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if "google.colab" in sys.modules:
    os.chdir("/content")
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found"
print("Ready.")

Working dir: /content/flyrank-ml-internship
Ready.


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## **Two paper findings + my methodology questions:**


**Finding 1: "What Predicts Growth?" — Logistic Regression, 71% holdout
accuracy** (ML Appendix, page 29). The paper reports this accuracy figure
describing which features separate growing from declining pages, with
Content Age as the strongest negative coefficient and Days Visible/recent
impressions as strongest positive.

**My methodology question (constructive tone):** The paper doesn't state
the base rate of growing vs. declining pages in this holdout set. Per the
hunting-leakage-and-validating skill, "accuracy of 71% on a label that is
62% positive is 9 points of skill, not 71" — so the honest read of this 71%
depends entirely on that missing number. If growing pages made up, say, 70%
of the sample, a model that just guessed "growing" every time would already
be close to 71%. I'd genuinely like to see the base rate reported next to
this accuracy figure, the same way the paper carefully reports base rates
elsewhere (e.g. the 82.89% figure in the search-volume myth section). This
isn't a claim the finding is wrong — it's a specific number needed to know
how much of the 71% is real signal.

**Finding 2: "What Predicts Health?" — Random Forest feature importance**
(ML Appendix, page 27), where Average Position (43%) and Impressions (32%)
dominate the importance ranking for predicting Health Score.

**My methodology question (constructive tone):** The paper is admirably
upfront that "the target itself is partly constructed from some of these
inputs, so importance is descriptive rather than causal" — Health Score is
literally defined as Impressions (30pts) + Position (30pts) + CTR (20pts) +
Scroll Depth (20pts) per the methodology page. Given that, I'd ask: was a
train-without-the-suspect-feature test run (per the leakage skill's
recommended check — train once with Position/Impressions, once without, and
see how much the score collapses)? The paper's own disclosure is exactly the
right instinct, but showing the actual size of the collapse would turn "this
is expected and not causal" from an assumption into a demonstrated number —
the same kind of before/after evidence I'm trying to produce for my own
model in Section 2 below.

*(Tone check: both questions ask "what number would resolve this," not
"this finding is invalid" — the paper already does a lot of the hard,
honest work here, including flagging its own health-score circularity and
demoting weak internal constructs; these questions are in that same spirit.)*

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
if "is_declining_label" not in df.columns:
    df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

numeric_features = [
    "search_volume", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
categorical_features = [
    "competition_level", "content_type", "main_intent",
    "age_tier", "freshness_tier", "word_count_tier",
    "impression_tier", "position_tier",
]

def build_pipeline():
    numeric_pipe = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
    categorical_pipe = Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))])
    preprocess = ColumnTransformer([("num", numeric_pipe, numeric_features), ("cat", categorical_pipe, categorical_features)])
    return Pipeline([("prep", preprocess), ("clf", LogisticRegression(max_iter=1000, random_state=42))])

def precision_at_k(y_true, scores, k=50):
    order = np.argsort(-scores)
    return np.array(y_true)[order][:k].mean()

X_cols = numeric_features + categorical_features

In [10]:
# --- BEFORE: naive random row split (the dishonest default) ---
X = df[X_cols]
y = df["is_declining_label"]

X_train_rand, X_test_rand, y_train_rand, y_test_rand = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

pipe_rand = build_pipeline()
pipe_rand.fit(X_train_rand, y_train_rand)
scores_rand = pipe_rand.predict_proba(X_test_rand)[:, 1]

p50_rand = precision_at_k(y_test_rand, scores_rand, k=50)
ap_rand = average_precision_score(y_test_rand, scores_rand)
base_rate_rand = y_test_rand.mean()

print("=== BEFORE: random row split ===")
print("Base rate:", round(base_rate_rand, 3))
print("Precision@50:", round(p50_rand, 3))
print("Average Precision:", round(ap_rand, 3))

=== BEFORE: random row split ===
Base rate: 0.542
Precision@50: 0.84
Average Precision: 0.708


In [11]:
# --- AFTER: honest client-grouped split ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

overlap = set(train_df["client_id"]) & set(test_df["client_id"])
print("Client overlap (should be 0):", len(overlap))

X_train_grp = train_df[X_cols]
y_train_grp = train_df["is_declining_label"]
X_test_grp = test_df[X_cols]
y_test_grp = test_df["is_declining_label"]

pipe_grp = build_pipeline()
pipe_grp.fit(X_train_grp, y_train_grp)
scores_grp = pipe_grp.predict_proba(X_test_grp)[:, 1]

p50_grp = precision_at_k(y_test_grp, scores_grp, k=50)
ap_grp = average_precision_score(y_test_grp, scores_grp)
base_rate_grp = y_test_grp.mean()

print("\n=== AFTER: client-grouped split ===")
print("Base rate:", round(base_rate_grp, 3))
print("Precision@50:", round(p50_grp, 3))
print("Average Precision:", round(ap_grp, 3))



Client overlap (should be 0): 0

=== AFTER: client-grouped split ===
Base rate: 0.517
Precision@50: 0.76
Average Precision: 0.583


In [12]:
# --- THE GAP TABLE (the actual finding) ---
comparison = pd.DataFrame({
    "split": ["random row split (naive)", "client-grouped split (honest)"],
    "base_rate": [base_rate_rand, base_rate_grp],
    "precision@50": [p50_rand, p50_grp],
    "average_precision": [ap_rand, ap_grp],
})
comparison.round(3)

,split,base_rate,precision@50,average_precision
0,random row split (naive),0.542,0.84,0.708
1,client-grouped split (honest),0.517,0.76,0.583


## **My model under an honest split (before/after)**

**What to look for after running:** if the random-split numbers are notably
higher than the grouped-split numbers, that gap IS the finding — it means
some of last week's apparent skill was the model recognizing which client a
page belongs to rather than learning a generalizable pattern. [After
running, fill in: state the actual gap, e.g. "Precision@50 dropped from X
under random split to Y under grouped split — a Z-point gap, meaning roughly
that much of the random-split score reflected client memorization rather
than genuine signal."] If the numbers are close, that's also worth reporting
honestly — it would suggest the model isn't leaning heavily on client-specific
patterns, which is reassuring, not disappointing.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [13]:
# --- Timeline check: are all features knowable before the label? ---
print("Features used (all should be pre-decision-point observable signals):")
print(X_cols)
print("\nExcluded (label-derived, would leak):", ["trend_direction", "trend_pct"])

Features used (all should be pre-decision-point observable signals):
['search_volume', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']

Excluded (label-derived, would leak): ['trend_direction', 'trend_pct']


In [14]:
# --- STRESS TEST: deliberately inject a leaky feature and confirm the harness catches it ---
# Per the skill: "Deliberately ADD a leaky feature and watch the score jump
# toward 1.0 — if it doesn't, your test harness itself is broken."

df_leaky = df.copy()
df_leaky["LEAKY_trend_direction_encoded"] = (df_leaky["trend_direction"].str.lower() == "down").astype(int)

leaky_features = X_cols + ["LEAKY_trend_direction_encoded"]

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx2, test_idx2 = next(gss2.split(df_leaky, groups=df_leaky["client_id"]))
train_leaky = df_leaky.iloc[train_idx2]
test_leaky = df_leaky.iloc[test_idx2]

numeric_pipe_leaky = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
categorical_pipe_leaky = Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))])
preprocess_leaky = ColumnTransformer([
    ("num", numeric_pipe_leaky, numeric_features + ["LEAKY_trend_direction_encoded"]),
    ("cat", categorical_pipe_leaky, categorical_features),
])
pipe_leaky = Pipeline([("prep", preprocess_leaky), ("clf", LogisticRegression(max_iter=1000, random_state=42))])
pipe_leaky.fit(train_leaky[leaky_features], train_leaky["is_declining_label"])
scores_leaky = pipe_leaky.predict_proba(test_leaky[leaky_features])[:, 1]

ap_leaky = average_precision_score(test_leaky["is_declining_label"], scores_leaky)
p50_leaky = precision_at_k(test_leaky["is_declining_label"], scores_leaky, k=50)

print("=== WITH deliberately injected leaky feature ===")
print("Precision@50:", round(p50_leaky, 3), "  Average Precision:", round(ap_leaky, 3))
print("\n=== Honest model (Section 2, no leak) for comparison ===")
print("Precision@50:", round(p50_grp, 3), "  Average Precision:", round(ap_grp, 3))

=== WITH deliberately injected leaky feature ===
Precision@50: 1.0   Average Precision: 1.0

=== Honest model (Section 2, no leak) for comparison ===
Precision@50: 0.76   Average Precision: 0.583


**Stress test result:** [After running, fill in: e.g. "Injecting a
near-identical encoding of trend_direction pushed Precision@50 from X to Y
and Average Precision toward 1.0 — confirming the test harness correctly
detects leakage when it's present. This is reassuring for my honest model:
since my real feature set produces nowhere near this score, I have direct
evidence it isn't relying on a hidden copy of the label."]

**Full attack checklist:**
- [x] Timeline drawn: all features (impressions, position, CTR, engagement,
  freshness, word count) are observable pre-decision-point signals from the
  trailing 90-day window; `trend_direction`/`trend_pct` (label-derived) are
  excluded.
- [x] No label-derived or sibling columns in the real feature set — confirmed
  by the stress test above showing a sharp score jump only when one is added.
- [x] No product flags (health_score, priority_score, action_type) used —
  these aren't shipped in the starter CSV at all, so there's nothing to
  accidentally include.
- [x] Split grouped by `client_id`, zero overlap confirmed (Section 2).
- [x] Base rate printed next to every metric (Section 2 table).
- [x] Top feature importance sanity-checked in ML-08 — no single feature
  exceeded ~15% of importance, consistent with no leakage.
- [x] Metrics computed out-of-fold, on the held-out test split only, never
  on training data.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## **Claim rewrite:**

**My boldest original claim (from ML-08):** "Logistic Regression is the
clear choice for this lane's actual decision... a good example of why
complexity isn't rewarded for its own sake."

**Rewritten in safe language:** Under a client-grouped validation split,
Logistic Regression showed the highest observed Precision@50 (0.700) among
the methods tested here, compared to a rule-based baseline (0.280) and a
Random Forest (0.500). This is a directional, decision-support finding on
this specific 30,000-row starter slice and proxy label — it suggests, but
does not prove, that a simpler model will keep outperforming on the full
warehouse release, where more data and a genuinely future-observed label may
change which method wins. I'm not claiming Logistic Regression is
universally better than Random Forest, only that it measured better on this
data, this split, and this metric.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.